In [ ]:
import pandas as pd
import numpy as np
import os
print(os.getcwd())
df = pd.read_csv('../Datasets/raw/Dataset.csv')
print('数据集形状',df.shape)
print('数据集预览:')
df.head()

# 数据清洗

In [23]:
for col in df.columns:
    print(f'列{col} 中的缺失值数: {df[col].isnull().sum()}')
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
print(f'{df.shape}, 初步处理后剩余{df.shape[0]}条数据')

列event_time 中的缺失值数: 0
列event_type 中的缺失值数: 0
列product_id 中的缺失值数: 0
列category_id 中的缺失值数: 0
列category_code 中的缺失值数: 48129
列brand 中的缺失值数: 21841
列price 中的缺失值数: 0
列user_id 中的缺失值数: 0
列user_session 中的缺失值数: 0
(92793, 9), 初步处理后剩余92793条数据


# 处理用于关联分析的数据

In [ ]:
#取出所有购买的数据，然后只取userID，productID，category_code, user_session列
purchase_record = df[df['event_type'] == 'purchase'].loc[:,['user_id','product_id','category_code','user_session']]
#按会话记录排序
purchase_record.sort_values('user_session',inplace=True)
purchase_record.set_index('user_id',inplace=True)
purchase_record.to_csv('../Datasets/processed/asso.csv',index = False)

# 时间字段提取

In [ ]:
df['event_time'] = df['event_time'].str.replace('UTC','')
df['event_time'] = pd.to_datetime(df['event_time'])
df['Day'] = df['event_time'].dt.day
df['Hour'] = df['event_time'].dt.hour # 0-based
df['Weekday'] = df['event_time'].dt.dayofweek  #0-based
df['Is_Weekend'] = (df['Weekday'] >= 5).astype(bool)
df['At_night'] = ((df['Hour'] <= 6) | (df['Hour'] >= 17))
df['During_the_day'] = ((df['Hour'] > 6)& (df['Hour'] < 17))
df.to_csv('../Datasets/processed/preprocessed.csv', index = False)
#抽样查看
df.iloc[0:30000:5000]

# 基于会话进行特征聚类

In [ ]:
# 按 user_session 分组，计算会话级统计特征

# 1. 计算会话内事件数、独立商品数、是否购买、购买总金额
session_group = df.groupby('user_session')

# 基础统计
session_features = session_group.agg(
    event_count=('event_type', 'count'),
    unique_products=('product_id', 'nunique'),
    has_purchase=('event_type', lambda x: ('purchase' in x.values)),
    cart_count=('event_type', lambda x: (x == 'cart').sum()),
    purchase_count=('event_type', lambda x: (x == 'purchase').sum())
).reset_index()

# 计算会话时长（分钟）
# 先获取每个会话的最小和最大时间
session_time = df.groupby('user_session')['event_time'].agg(['min', 'max'])
session_time['session_duration_min'] = (session_time['max'] - session_time['min']).dt.total_seconds() / 60.0
session_features = session_features.merge(session_time[['session_duration_min']], on='user_session', how='left')

# 计算购买总金额（仅针对有购买行为的会话）
# 筛选购买记录，按会话聚合金额
purchase_amount = df[df['event_type'] == 'purchase'].groupby('user_session')['price'].sum().rename('purchase_amount')
session_features = session_features.merge(purchase_amount, on='user_session', how='left')
session_features['purchase_amount'] = session_features['purchase_amount'].fillna(0)

# 判断是否包含 cart → purchase 行为序列（按时间顺序）
def has_cart_before_purchase(group):
    """
    检查会话内是否存在 cart 事件发生在 purchase 事件之前
    """
    # 按时间排序
    group_sorted = group.sort_values('event_time')
    events = group_sorted['event_type'].tolist()
    # 查找第一个 purchase 的位置
    try:
        first_purchase_idx = events.index('purchase')
    except ValueError:
        return False
    # 检查在第一个 purchase 之前是否有 cart
    return 'cart' in events[:first_purchase_idx]

# 应用函数到每个会话
cart_to_purchase = df.groupby('user_session').apply(has_cart_before_purchase).reset_index(name='cart_to_purchase')
session_features = session_features.merge(cart_to_purchase, on='user_session', how='left')

# 可选：会话内浏览的不同品类数（基于 category_code）
unique_categories = df.groupby('user_session')['category_code'].nunique().rename('unique_categories')
session_features = session_features.merge(unique_categories, on='user_session', how='left')

# 展示结果
print("会话级特征表形状:", session_features.shape)
session_features.to_csv('../Datasets/processed/session_based_features.csv', index=False)